In [1]:
!pip install transformers datasets
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, pipeline, DataCollatorForLanguageModeling, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
import tokenizers
import numpy as np
import matplotlib.pyplot as plt
import re
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

PyTorch device: cuda


# Stage 1

In [2]:
def generate_exprs():
    for a in range(2, 20 + 1):
        for b in range(2, 20 + 1):
            for c in range(2, 20 + 1):
                yield {
                    "expr": f"({a} + {b}) - {c}",
                    "result": (a + b) - c,
                    "reasoning": f"{a} + {b} = {a + b}. {a + b} - {c} = {(a + b) - c}."
                }
                if a >= b:
                    yield {
                        "expr": f"({a} - {b}) + {c}",
                        "result": (a - b) + c,
                        "reasoning": f"{a} - {b} = {a - b}. {a - b} + {c} = {(a - b) + c}."
                    }
                yield {
                    "expr": f"{a} + ({b} * {c})",
                    "result": a + (b * c),
                    "reasoning": f"{b} * {c} = {b * c}. {a} + {b * c} = {a + (b * c)}."
                }
                yield {
                    "expr": f"({a} * {b}) - c",
                    "result": (a * b) - c,
                    "reasoning": f"{a} * {b} = {a * b}. {a * b} - {c} = {(a * b) - c}."
                }
def convert_to_instructions(row):
    row["prompt"] = "Instruction: Solve the arithmetic expression step by step. Use exactly two lines:"
    row["prompt"] += "\nReasoning: ..."
    row["prompt"] += "\nAnswer: <integer>"
    row["prompt"] += f"\nExpression: {row["expr"]}"
    row["prompt"] += "\nResponse"

    row["response"] = row["prompt"]
    row["response"] += f"\nReasoning: {row["reasoning"]}"
    row["response"] += f"\nAnswer: {row["result"]}"
    return row

# hf's train_test_split cannot be used to create multiple split
# So create them manually
dataset = Dataset.from_generator(generate_exprs).map(convert_to_instructions).shuffle(seed=42)
dataset = DatasetDict({ "all": dataset })
SPLITS = [
    ("train_sft", 800),
    ("val_sft", 200),
    ("train_ppo", 100),
    ("test", 100)
]
skip_count = 0
for name, amount in SPLITS:
    dataset[name] = dataset["all"].skip(skip_count).take(amount)
    skip_count += amount
del SPLITS, skip_count, name, amount
del dataset["all"]
assert sum(map(len, dataset.values())) == 1200
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/24187 [00:00<?, ? examples/s]

DatasetDict({
    train_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 800
    })
    val_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 200
    })
    train_ppo: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 100
    })
    test: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 100
    })
})


# Stage 2

In [3]:
base_name = "distilgpt2"
tok = AutoTokenizer.from_pretrained(base_name, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(base_name).to(device)
base_model.config.pad_token_id = tok.pad_token_id

max_len = 1024

def tokenize_sft(ex):
    prompt_ids = tok(ex["prompt"], truncation=True, max_length=max_len, padding=False)["input_ids"]
    full = tok(ex["response"], truncation=True, max_length=max_len, padding=False)

    input_ids = full["input_ids"]
    attn = full["attention_mask"]

    labels = input_ids.copy()
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels} | ex

tokenized_dataset = dataset.map(tokenize_sft)
print(tokenized_dataset)

padder = DataCollatorWithPadding(tokenizer=tok)
def collate(features):
    prompts = [f["prompt"] for f in features]
    feats = [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features]
    batch = padder(feats)

    maxT = batch["input_ids"].shape[1]
    labels = []
    for f in features:
        lab = f["labels"]
        if len(lab) < maxT:
            lab = lab + [-100]*(maxT - len(lab))
        else:
            lab = lab[:maxT]
        labels.append(lab)

    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    batch["prompt"] = prompts
    return batch

Parameter 'function'=<function tokenize_sft at 0x7ffcf32d1620> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
    val_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    train_ppo: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
    test: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})


In [4]:
sft_model = AutoModelForCausalLM.from_pretrained(base_name).to(device)
sft_model.config.pad_token_id = tok.pad_token_id

args = TrainingArguments(
    output_dir="sft_ckpt",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch ~32
    num_train_epochs=10,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=500,                  # ensures we save at least once
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=sft_model,
    args=args,
    train_dataset=tokenized_dataset["train_sft"],
    eval_dataset=tokenized_dataset["val_sft"],
    data_collator=collate
)

trainer.train()
trainer.save_model("./sft_model")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,0.860900,0.463344
100,0.441200,0.406710
150,0.387400,0.362458
200,0.358200,0.347514
250,0.339300,0.338017


# Stage 3

In [5]:
def tokenize_ppo(ex):
    input_ids = torch.tensor(tok(ex["prompt"], truncation=True, max_length=max_len, padding=False)["input_ids"])

    return {"input_ids": input_ids, "attention_mask": torch.ones_like(input_ids), "labels": input_ids } | ex

ppo_dataset = dataset.map(tokenize_ppo)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [6]:
class PPOModel(nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.lm = AutoModelForCausalLM.from_pretrained(base_model_name)
        hidden = self.lm.config.n_embd
        self.value_head = nn.Linear(hidden, 1)
    def forward(self, input_ids, attention_mask):
        out = self.lm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        last_hidden = out.hidden_states[-1]                # [B, T, H]
        values = self.value_head(last_hidden).squeeze(-1)  # [B, T]
        return out.logits, values
def logprobs_from_logits(logits, labels):
    logp = torch.log_softmax(logits, dim=-1)
    return torch.gather(logp, 2, labels.unsqueeze(-1)).squeeze(-1)
def generate_response(model, prompt_ids, max_new_tokens=64):
    return model.lm.generate(
        prompt_ids,
        max_new_tokens=max_new_tokens,
        pad_token_id=tok.eos_token_id
    )
def compute_reward(text, ground_truth):
    match = re.search(r"Answer:\s*(-?\d+)", text)
    if match is None:
        return 0.0
    pred = int(match.group(1))
    return 1.0 if pred == ground_truth else 0.2
def compute_advantages(rewards, values, gamma=0.99, lam=0.95):
    adv = torch.zeros_like(rewards)
    last = 0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] - values[t]
        last = delta + gamma * lam * last
        adv[t] = last
    returns = adv + values
    return adv, returns
def ppo_loss(
    logp, logp_old, advantages,
    values, returns,
    kl, clip=0.2,
    vf_coef=0.5,
    kl_coef=0.1
):
    ratio = torch.exp(logp - logp_old)
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1 - clip, 1 + clip) * advantages
    policy_loss = -torch.min(unclipped, clipped).mean()

    value_loss = vf_coef * (returns - values).pow(2).mean()
    kl_loss = kl_coef * kl.mean()

    return policy_loss + value_loss + kl_loss

In [7]:
ppo_model = PPOModel("./sft_model").to(device)
ref_model = AutoModelForCausalLM.from_pretrained("./sft_model").to(device)
ref_model.requires_grad_(False)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
optimizer = torch.optim.AdamW(ppo_model.parameters(), lr=1e-5)
ppo_model.train()

ppo_batch_size = 8
loss_accum = 0.0
advantages_buffer = []

for step, ex in enumerate(dataset["train_ppo"]):
    prompt = ex["prompt"]
    answer = ex["result"]

    prompt_ids = tok(prompt, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        full_ids = generate_response(ppo_model, prompt_ids)
        ref_logits = ref_model(full_ids).logits

    logits, values_seq = ppo_model(
        full_ids,
        attention_mask=torch.ones_like(full_ids)
    )

    # token logprobs
    logp = logprobs_from_logits(logits[:, :-1], full_ids[:, 1:])
    logp_ref = logprobs_from_logits(ref_logits[:, :-1], full_ids[:, 1:])

    # mask prompt tokens
    prompt_len = prompt_ids.shape[1]
    logp = logp[:, prompt_len - 1:]
    logp_ref = logp_ref[:, prompt_len - 1:]
    values_seq = values_seq[:, prompt_len - 1:]

    decoded = tok.decode(full_ids[0], skip_special_tokens=True)
    reward_val = compute_reward(decoded, answer)
    reward = torch.tensor([reward_val], device=device)

    # bootstrap value
    values_last = values_seq[:, -1]

    advantage = reward - values_last.detach()
    advantages_buffer.append(advantage)

    # PPO losses (token‑level)
    ratio = torch.exp(logp - logp_ref)
    advantages_t = advantage.unsqueeze(-1)

    clip = 0.2
    vf_coef = 0.5
    kl_coef = 0.05

    policy_loss = -torch.min(
        ratio * advantages_t,
        torch.clamp(ratio, 1 - clip, 1 + clip) * advantages_t
    ).mean()

    returns = reward.expand_as(values_last)
    value_loss = vf_coef * (returns - values_last).pow(2).mean()

    kl = logp - logp_ref
    kl_loss = kl_coef * kl.mean()

    loss = policy_loss + value_loss + kl_loss
    loss = loss / ppo_batch_size
    loss.backward()

    if (step + 1) % ppo_batch_size == 0:
        # normalize advantages *across the batch*
        adv = torch.cat(advantages_buffer)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        advantages_buffer.clear()

        optimizer.step()
        optimizer.zero_grad()

    print(f"Step {step} | Reward {reward_val:+.2f}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Step 0 | Reward +0.20
Step 1 | Reward +0.20
Step 2 | Reward +1.00
Step 3 | Reward +0.20
Step 4 | Reward +0.20
Step 5 | Reward +0.20
Step 6 | Reward +0.20
Step 7 | Reward +0.20
Step 8 | Reward +0.20
Step 9 | Reward +0.20
Step 10 | Reward +0.20
Step 11 | Reward +0.20
Step 12 | Reward +0.20
Step 13 | Reward +0.20
Step 14 | Reward +0.20
Step 15 | Reward +0.20
Step 16 | Reward +0.20
Step 17 | Reward +0.20
Step 18 | Reward +0.20
Step 19 | Reward +0.20
Step 20 | Reward +0.20
Step 21 | Reward +0.20
Step 22 | Reward +0.20
Step 23 | Reward +0.20
Step 24 | Reward +0.20
Step 25 | Reward +0.20
Step 26 | Reward +0.20
Step 27 | Reward +0.20
Step 28 | Reward +0.20
Step 29 | Reward +0.20
Step 30 | Reward +0.20
Step 31 | Reward +0.20
Step 32 | Reward +0.20
Step 33 | Reward +0.20
Step 34 | Reward +0.20
Step 35 | Reward +0.20
Step 36 | Reward +0.20
Step 37 | Reward +0.20
Step 38 | Reward +0.20
Step 39 | Reward +0.20
Step 40 | Reward +0.20
Step 41 | Reward +0.20
Step 42 | Reward +0.20
Step 43 | Reward +0.2

# Stage 4

In [9]:
ANSWER_PATTERN = re.compile(r"Answer:\s*(-?\d+)")

def evaluate_model(model, tokenizer, test_dataset):
    model.eval()
    correct = 0
    format_adhered = 0
    total_reward = 0
    debug = []

    for row in test_dataset:
        prompt = row["prompt"]
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=50,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        full = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        response = full[len(prompt):]
        reward = compute_reward(response, row["result"])
        total_reward += reward
        m = ANSWER_PATTERN.search(response)
        debug.append({
            "expr": row["expr"],
            "result": row["result"],
            "answer": m.group(1) if m else "<NOT FOUND>",
            "adhered": True if m else False,
            "full": full,
            "response": response,
            "reward": reward
        })
        if m:
            format_adhered += 1
            if int(m.group(1)) == int(row["result"]):
                correct += 1

    print(f"\tAccuracy: {correct / len(test_dataset):.2%}")
    print(f"\tAdherence: {format_adhered / len(test_dataset):.2%}")
    print(f"\tReward: {total_reward / len(test_dataset):.2f}")
    return debug


In [10]:
eval_debug = {}
print("Base:")
eval_debug["base"] = evaluate_model(base_model, tok, dataset["test"])
print("\tTest Prompts:")
for d in eval_debug["base"][42:42+5]:
    print(f"\t\t{d["full"]}")
print("SFT:")
eval_debug["sft"] = evaluate_model(sft_model, tok, dataset["test"])
print("\tTest Prompts:")
for d in eval_debug["base"][67:67+5]:
    print(f"\t\t{d["full"]}")
print("PPO:")
eval_debug["ppo"] = evaluate_model(ppo_model.lm, tok, dataset["test"])
print("\tTest Prompts:")
for d in eval_debug["base"][10:10+5]:
    print(f"\t\t{d["full"]}")
with open("debug.json", "w") as f:
    json.dump(eval_debug, f)

Base:
	Accuracy: 0.00%
	Adherence: 0.00%
	Reward: 0.00
	Test Prompts:
		Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: 17 + (13 * 20)
Response: <integer>
Expression: 17 + (13 * 20)
Response: <integer>
Expression: 17 + (13 * 20)
Response: <integer>
Expression: 17 + (13 * 20)

		Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: (7 + 3) - 11
Response: <integer>
Answer: <integer>
Answer: <integer>
Answer: <integer>
Answer: <integer>
Answer: <integer>
Answer: <integer>
Answer: <integer>
Answer: <
		Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: (18 - 14) + 16
Response: <integer>
Expression: (18 - 14) + 16
Response: <integer>
Expression: (18 - 14) + 16
Response: <integer>
Expression: (18 - 14) + 16

		Instruction: Solve the arithmetic expression step by step.

# Stage 5

In [11]:
PROBLEMS = [
  {
    "problem": "Liam had 7 apples and picked 5 more from a tree. After giving 6 apples to his friend, how many apples does he have left?",
    "answer": 6
  },
  {
    "problem": "A library had 9 new books arrive and later received 8 donated books. If 5 books were checked out, how many books remain?",
    "answer": 12
  },
  {
    "problem": "Emma baked 4 muffins in the morning and 10 more in the afternoon. If her family ate 7 muffins, how many muffins are left?",
    "answer": 7
  },
  {
    "problem": "There were 15 birds on a fence and 3 more joined them. If 8 birds flew away, how many birds are still on the fence?",
    "answer": 10
  },
  {
    "problem": "Noah collected 6 shells on the beach and then found 14 more. After losing 9 shells, how many shells does he have?",
    "answer": 11
  },
  {
    "problem": "A farmer had 12 cows in a field. After selling 5 cows and then buying 4 more, how many cows does the farmer have?",
    "answer": 11
  },
  {
    "problem": "There were 20 students in a class. If 11 students left for an activity and 3 returned, how many students are in the class now?",
    "answer": 12
  },
  {
    "problem": "Olivia saved 9 dollars and spent 2 dollars. Later, she earned 8 more dollars. How much money does she have now?",
    "answer": 15
  },
  {
    "problem": "A store had 16 shirts in stock. After selling 7 shirts and receiving 6 new ones, how many shirts are in the store?",
    "answer": 15
  },
  {
    "problem": "Jack had 10 toy cars. He gave 4 to his brother and then got 9 new toy cars for his birthday. How many toy cars does he have now?",
    "answer": 15
  },
  {
    "problem": "A teacher has 3 boxes of pencils, with 4 pencils in each box, and also has 3 extra pencils. How many pencils does the teacher have in total?",
    "answer": 23
  },
  {
    "problem": "There are 2 bags with 6 oranges in each bag. If there are also 8 loose oranges, how many oranges are there altogether?",
    "answer": 20
  },
  {
    "problem": "A bookshelf holds 3 stacks of books, with 4 books in each stack, plus 7 extra books. How many books are on the shelf?",
    "answer": 19
  },
  {
    "problem": "A gardener planted 6 rows of flowers with 3 flowers in each row, and then added 5 more flowers. How many flowers are there?",
    "answer": 23
  },
  {
    "problem": "There are 2 packages with 5 cookies each, and 9 cookies on a plate. How many cookies are there in total?",
    "answer": 19
  },
  {
    "problem": "A baker made 4 trays of cookies, with 5 cookies on each tray. If 6 cookies were eaten, how many cookies remain?",
    "answer": 14
  },
  {
    "problem": "There are 7 boxes with 3 toys in each box. If 2 toys are broken and removed, how many toys are left?",
    "answer": 19
  },
  {
    "problem": "A classroom has 6 rows of desks with 4 desks in each row. If 5 desks are taken out for repairs, how many desks remain?",
    "answer": 19
  },
  {
    "problem": "A factory produces 5 packs of batteries, with 8 batteries in each pack. If 10 batteries are used, how many batteries are left?",
    "answer": 30
  },
  {
    "problem": "There are 9 plates with 3 sandwiches on each plate. If 7 sandwiches are eaten, how many sandwiches remain?",
    "answer": 20
  }
]

In [12]:
def generate(model, prompt, tokenizer, max_tokens=100):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)[len(prompt):]

In [13]:
def calculator_tool(*args):
    if len(args) != 1:
        return f"<error: incorrect number of arguments; expected 1, got {len(args)}>"
    expression = expression.replace(" ", "")
    expression = expression.replace("\n", "")
    expression = expression.replace("\r", "")
    expression = expression.replace("\t", "")
    # Since reasoning steps in dataset end in periods, strip them from input just in case
    if expression[-1] == ".":
        expression = expression[:-1]

    # Validate
    allowed = "0123456789("
    depth = 0
    for c in expression:
        if c not in allowed:
            return f"<error: illegal char, expected one of: {allowed}>"
        elif c == "(":
            depth += 1
            allowed = "0123456789("
        elif c == ")":
            depth -= 1
            allowed = ")+-*/"
            if depth < 0:
                return "<error: too many parenthesis>"
        elif c in "01234567890":
            allowed = "0123456789+-*/"
        elif c in "+-*/":
            allowed = "0123456789("
        else:
            assert False, "Unreachable"
    
    if depth != 0:
        return "<error: too few parenthesis>"
    
    try:
        # this is hacky and horribly insecure but we do validate the input so it should be fine
        return f"Result: {eval(expression)}"
    except Exception:
        return "<error: failed to evaluate>"

In [14]:
def run_tool_agent(model, tokenizer, problem):
    SYSTEM_PROMPT = f"""
    You are a calculator agent tasked with solving math problems using provided tools.
    
    You must think step-by-step, but output ONLY one JSON object each turn.
    
    Output Schema:
    {{ "reason": "...", "tool": "calculator|finish", "args": [ ... ] }}
    
    Rules:
    - Provide your reasoning in the reason field
    - Use the calculator tool to perform calculations. It accepts one argument; a math expression to evaluate (string)
    - Use the finish tool once you have solved the problem. It accepts one argument; your final answer (integer)
    
    The problem you need to solve is:
    {problem["problem"]}
    """
    log = SYSTEM_PROMPT
    tool_log = []
    final_answer = None
    for _ in range(10):
        output = generate(model, log, tok, 50)
        log += f"\n{output}\n"
        try:
            m = re.search(r"\{.*\}", output, flags=re.DOTALL)
            tool = json.loads(None if not m else m.group(0))
        except Exception:
            continue
        print(tool)
        tool_log.append(tool)

        args = tool["args"]
        
        if tool["tool"] == "finish":
            if len(args) != 1:
                log += f"\n<rror: finish tool expects 1 argument; got {len(args)}>"
                continue
            final_answer = args[0]
            break
        elif tool["tool"] == "calculator":
            log += f"\n{calculator_tool(*args)}\n"
        else:
            log += f"<error: unknown tool '{tool["tool"]}'>"
    try:
        return int(final_answer) == problem["answer"], final_answer, tool_log
    except Exception:
        return False, "<none>", tool_log

In [15]:
ppo_correct = 0
tool_correct = 0
for p in PROBLEMS:
    prompt = f"""
    What is the answer the following math problem: '{p["problem"]}'
    Give your answer in one line, using the following format:
    Answer: <integer>
    """
    response = generate(ppo_model.lm, prompt, tok, 10).strip()
    m = ANSWER_PATTERN.search(response)
    was_correct = m and int(m.group(1)) == p["answer"]
    if was_correct:
        ppo_correct += 1
    print(f"\t{p["problem"]}")
    print(f"\t\tPPO: {response} {"✅" if was_correct else "❌"}")

    was_correct, ans, tool_log = run_tool_agent(ppo_model.lm, tok, p)
    print(f"\t\tTools: {ans} {"✅" if was_correct else "❌"}")
    for t in tool_log:
        print(f"\t\t\t{json.dumps(t)}")
    
print(f"PPO Accuracy: {ppo_correct / len(PROBLEMS):.2%}")
print(f"Tool Accuracy: {tool_correct / len(PROBLEMS):.2%}")

	Liam had 7 apples and picked 5 more from a tree. After giving 6 apples to his friend, how many apples does he have left?
		PPO: Answer: 5 = 10. ❌
		Tools: <none> ❌
	A library had 9 new books arrive and later received 8 donated books. If 5 books were checked out, how many books remain?
		PPO: Answer: 8 = 22. ❌
		Tools: <none> ❌
	Emma baked 4 muffins in the morning and 10 more in the afternoon. If her family ate 7 muffins, how many muffins are left?
		PPO: Answer: 10 = 20. 20 = -7 ❌
		Tools: <none> ❌
	There were 15 birds on a fence and 3 more joined them. If 8 birds flew away, how many birds are still on the fence?
		PPO: Answer: 10. 10 + 20 = 30. ✅
		Tools: <none> ❌
	Noah collected 6 shells on the beach and then found 14 more. After losing 9 shells, how many shells does he have?
		PPO: Answer: 9. 9 + 9 = 21. ❌
		Tools: <none> ❌
	A farmer had 12 cows in a field. After selling 5 cows and then buying 4 more, how many cows does the farmer have?
		PPO: Answer: 2nd Place = 0. 2nd ❌
		Tools: 

# Stage 6

## a

When PPO was used, the accuracy remained the same.

## b

The $0.2$ reward value helps the model learn to format its responses in the desired format. Without it, the model would be able to learn if it is wrong because the answer was wrong or because it was in the wrong format.

## c

Adding PPO kepts the format adherence the same. However, if the adherence after SFT was lower than 100%, then PPO likely would have increased it since the reward function gives $+0.2$ if the format is respected, which would intice the model to output `Answer: ` more frequently.

## d

Test example: `16 + (11 * 13)`
> Answer: 159

Base:
```
: <integer>
Expression: 16 + (11 * 13)
Response: <integer>
Expression: 16 + (11 * 13)
Response: <integer>
Expression: 16 + (11 * 13)

```
SFT:
```

Reasoning: 11 * 13 = 136. 16 + 136 = 137.
Answer: 137 = 138.
Answer: 138 = 139.
Answer: 140 = 140.
Answer: 141 = 141.
Answer: 142 = 142
```
PPO:
```

Reasoning: 11 * 13 = 136. 16 + 136 = 138.
Answer: 138 - 138 = 141.
Answer: 141 - 138 = 142.
Answer: 142 - 138 = 143.
Answer: 143 - 138 =
```

The base model did not follow the instructions at all.

When SFT was added, it began following the instructions, but it got the wrong answer and also started counting up.

When PPO was added, it got basically the same answer but the counting was now a subtraction.

Both the SFT and PPO models learned the correct order of operations, but they did not give the correct answers.



## e

The model was unable to successfuly call any tool, so it never got any answers.